# CIFAR-10: Building and Training a CNN

**Prepared by Fonyuy Gita — SEED AI Bootcamp, Deep Learning Module (CNN track)**

This notebook loads the prepared CIFAR-10 arrays and focuses entirely on
the network: what a CNN is and why it works differently from the ANNs
we've built so far, building it, training it, and evaluating it.

Run `01_cifar10_data_preparation.ipynb` first if you haven't already.

**What this notebook covers:**
1. Why CNNs exist, and the three ideas behind them
2. Load the prepared image arrays
3. Data augmentation
4. Build the CNN
5. Train it
6. Visualize training
7. Evaluate on the test set
8. Save the model and run inference on a sample image


## Step 1: Why a CNN, not an ANN, for images?

Every ANN we've built so far (diabetes, malaria, heart disease) treated
its input as a flat list of independent numbers, glucose, BMI, age, order
didn't matter, each feature was its own column. That works fine for
tabular data.

An image is different: **which pixels are next to which other pixels
matters enormously.** A cat's ear only means something in relation to the
pixels immediately around it. If you fed a 32x32x3 image into a plain
Dense layer, you'd flatten it into 3,072 independent numbers and throw
away every spatial relationship between them, the network would have to
relearn "nearby pixels are related" completely from scratch, for every
single position in the image, with no shortcuts.

CNNs solve this with three ideas:

```mermaid
flowchart LR
    A["1. Convolution\na small filter slides\nacross the image"] --> B["2. Pooling\ndownsample, keep\nthe strongest signal"]
    B --> C["3. Hierarchy\nstack many layers:\nedges to shapes to objects"]
```

**Convolution:** instead of one giant Dense layer connecting every pixel
to every neuron, a small filter (say, 3x3 pixels) slides across the
entire image, computing the same small calculation at every position.
The same filter weights are reused everywhere, called **weight sharing**,
so a filter that learns to detect a vertical edge in the top-left corner
automatically detects vertical edges everywhere else in the image too, no
retraining required per position.

**Pooling:** after detecting features, we shrink the image by keeping
only the strongest signal in each small region (typically the maximum
value in each 2x2 block). This reduces computation and makes the network
care less about a feature's exact pixel position and more about whether
it's present nearby.

**Hierarchy:** stack several convolution + pooling blocks. Early layers
learn simple things (edges, colors, gradients). Middle layers combine
those into textures and shapes. Late layers combine those into object
parts and, eventually, full objects. No one programs this hierarchy in,
it emerges from stacking the same simple operation repeatedly and
training with backpropagation, exactly the same gradient descent and
chain rule mechanics from the ANN projects, just applied to a different
kind of layer.


### How one convolution actually works

```mermaid
flowchart LR
    subgraph Image patch 3x3
    A1["1 0 1"]
    A2["0 1 0"]
    A3["1 0 1"]
    end
    subgraph Filter 3x3
    B1["1 0 -1"]
    B2["1 0 -1"]
    B3["1 0 -1"]
    end
    A1 & A2 & A3 --> D[element-wise multiply, then sum]
    B1 & B2 & B3 --> D
    D --> E["one output number"]
```

The filter slides to the next position and repeats. A 3x3 filter sliding
across a 32x32 image (with padding to keep the size the same) produces a
full 32x32 grid of these output numbers, one per position, called a
**feature map**. A single Conv2D layer learns many filters at once (32,
64, 128, whatever you choose), each producing its own feature map, each
specializing in detecting a different pattern.


## Step 2: Import libraries

In [ ]:
import json

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import confusion_matrix, classification_report

tf.random.set_seed(42)
np.random.seed(42)


## Step 3: Load the prepared data

In [ ]:
X_train = np.load('cifar_X_train.npy')
y_train = np.load('cifar_y_train.npy')
X_val = np.load('cifar_X_val.npy')
y_val = np.load('cifar_y_val.npy')
X_test = np.load('cifar_X_test.npy')
y_test = np.load('cifar_y_test.npy')

with open('cifar_classes.json') as f:
    CLASSES = json.load(f)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")
print(f"Classes: {CLASSES}")


## Step 4: Data augmentation

With only 45,000 training images, the network can start memorizing
specific images rather than learning general features, the same
overfitting problem we saw in the ANN projects, just more severe here
because images have so much more room to memorize.

**Data augmentation** fights this by randomly transforming each training
image a little differently every epoch, flipping it, rotating it
slightly, zooming in a bit. The network never sees the exact same image
twice, which forces it to learn genuine features ("has whiskers") rather
than memorized specifics ("this exact cat photo"). Augmentation is only
ever applied to training data, never to validation or test data, we want
to evaluate on real, unmodified images.

In [ ]:
augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.06),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

# Preview what augmentation does to one image
sample_image = X_train[0:1]
fig, axes = plt.subplots(1, 6, figsize=(14, 3))
axes[0].imshow(sample_image[0])
axes[0].set_title('Original')
axes[0].axis('off')

for i in range(1, 6):
    augmented = augmentation(sample_image, training=True)
    axes[i].imshow(augmented[0].numpy())
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')

plt.suptitle(f'Data Augmentation Preview — {CLASSES[y_train[0]]}', fontsize=12)
plt.tight_layout()
plt.savefig('cifar_augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 5: Build the CNN

We stack three convolutional blocks, each one shrinking the image while
growing the number of filters, followed by a small classifier head.

```mermaid
flowchart TD
    A["Input 32x32x3"] --> B["Block 1: Conv2D x2 (32 filters) + Pool, 32x32x3 -> 16x16x32"]
    B --> C["Block 2: Conv2D x2 (64 filters) + Pool, 16x16x32 -> 8x8x64"]
    C --> D["Block 3: Conv2D x2 (128 filters) + Pool, 8x8x64 -> 4x4x128"]
    D --> E["Flatten, 4x4x128 -> 2048"]
    E --> F["Dense 512 + Dropout"]
    F --> G["Dense 10, softmax, class probabilities"]
```

Notice the pattern: as we go deeper, the image gets **spatially smaller**
(32 to 16 to 8 to 4) but **channel-wise deeper** (3 to 32 to 64 to 128).
Early layers see a large area with few filters (simple features like
edges), later layers see a small, already-abstracted area with many
filters (complex, composed features). This trade-off, shrinking space
while growing depth, is the standard CNN design pattern.


In [ ]:
def build_cifar_cnn(num_classes=10):
    inputs = keras.Input(shape=(32, 32, 3), name='image')

    # Data augmentation lives inside the model, so it only activates
    # during training (training=True), and is automatically skipped
    # during evaluate() and predict()
    x = augmentation(inputs)

    # Block 1: 32x32x3 -> 16x16x32
    x = keras.layers.Conv2D(32, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Conv2D(32, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.SpatialDropout2D(0.2)(x)

    # Block 2: 16x16x32 -> 8x8x64
    x = keras.layers.Conv2D(64, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Conv2D(64, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.SpatialDropout2D(0.3)(x)

    # Block 3: 8x8x64 -> 4x4x128
    x = keras.layers.Conv2D(128, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Conv2D(128, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.SpatialDropout2D(0.4)(x)

    # Classifier head: 4x4x128 -> 10
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(512, activation='relu')(x)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax', name='class_probs')(x)

    return keras.Model(inputs, outputs, name='CIFARNet')


model = build_cifar_cnn()
model.summary()


A quick note on the output layer: this is **multi-class** classification
(10 classes, not 2), so instead of a single sigmoid neuron, we use 10
neurons with a **softmax** activation. Softmax forces all 10 outputs to
be positive and sum to exactly 1.0, so they can be read directly as class
probabilities.

In [ ]:
dummy = tf.random.normal([4, 32, 32, 3])
output = model(dummy, training=False)
print(f"Output shape: {output.shape}")
print(f"Each row sums to 1.0 (softmax): {output.numpy().sum(axis=1)}")
print(f"Total parameters: {model.count_params():,}")


## Step 6: Compile the model

We use `sparse_categorical_crossentropy` rather than the
`binary_crossentropy` from the earlier ANN projects, since we now have
10 classes, not 2. "Sparse" means our labels are plain integers (0-9)
rather than one-hot encoded vectors, Keras handles the conversion
internally.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


## Step 7: Train the model

Same callback family as the ANN projects: stop early once validation
accuracy stops improving, shrink the learning rate when progress stalls,
keep only the best checkpoint. Training a CNN on real images takes
noticeably longer per epoch than the small ANNs we trained before, this
is expected, convolution is more computationally expensive than a Dense
layer.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15,
        restore_best_weights=True, mode='max', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=5,
        mode='max', min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(
        'best_cifar_model.keras', monitor='val_accuracy',
        save_best_only=True, mode='max', verbose=0),
]

history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
)

print(f"\nEpochs run: {len(history.history['loss'])}")
print(f"Best val accuracy: {max(history.history['val_accuracy']):.4f}")


## Step 8: Visualize training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle('CIFAR-10 Training History', fontsize=13)
plt.tight_layout()
plt.savefig('cifar_training_history.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 9: Evaluate on the test set

In [ ]:
model.load_weights('best_cifar_model.keras')

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_prob = model.predict(X_test, verbose=0)
y_pred = y_prob.argmax(axis=1)


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('cifar_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, y_pred, target_names=CLASSES))


With 10 classes, look at the confusion matrix for which pairs of classes
the model mixes up most. Cats and dogs, or automobiles and trucks, are
typically the hardest pairs, they genuinely share more visual features
with each other than with, say, an airplane.

## Step 10: Save the model

In [ ]:
model.save('cifar10_cnn.keras')
print("Saved cifar10_cnn.keras")


## Step 11: Run inference on a sample image

In [ ]:
def predict_image(image, model, classes):
    """image: a single image array, shape (32, 32, 3), values in [0, 1]"""
    batch = image[np.newaxis, ...]  # add batch dimension: (1, 32, 32, 3)
    probabilities = model.predict(batch, verbose=0)[0]
    predicted_idx = probabilities.argmax()

    plt.figure(figsize=(3, 3))
    plt.imshow(image)
    plt.title(f"Predicted: {classes[predicted_idx]} ({probabilities[predicted_idx]:.1%})")
    plt.axis('off')
    plt.show()

    top3_idx = probabilities.argsort()[-3:][::-1]
    print("Top 3 predictions:")
    for idx in top3_idx:
        print(f"  {classes[idx]:<12} {probabilities[idx]:.1%}")


# Try it on a random test image
sample_idx = np.random.randint(len(X_test))
print(f"Actual class: {CLASSES[y_test[sample_idx]]}")
predict_image(X_test[sample_idx], model, CLASSES)


### Done

We went from a raw `.tar.gz` file, through unpickling, reshaping,
normalizing, and augmenting, to a trained CNN that classifies real color
photographs into 10 categories. The core loop, forward pass, loss,
backward pass, weight update, is exactly the same as every ANN we
trained earlier, only the layer types changed to match the shape of the
data.